In [4]:
console.log("Hello World")


Hello World


In [7]:
const token = Deno.env.get("GITHUB_TOKEN");
if (!token) throw new Error("GITHUB_TOKEN environment variable is not set.");

const apiBase = Deno.env.get("GITHUB_MODELS_URL") ?? "https://models.github.ai";
const endpoint = `${apiBase}/inference/chat/completions`;

const model = Deno.env.get("GITHUB_MODEL") ?? "openai/gpt-4.1";
const question = "What is the capital of France?";

const payload = {
  model,
  messages: [{ role: "user", content: question }]
};

const resp = await fetch(endpoint, {
  method: "POST",
  headers: {
    "Authorization": `Bearer ${token}`,
    "Accept": "application/json",
    "Content-Type": "application/json",
    "X-GitHub-Api-Version": "2022-11-28"
  },
  body: JSON.stringify(payload),
});

if (!resp.ok) {
  const text = await resp.text();
  console.error(`Request failed (${resp.status}): ${text}`);
} else {
  const json = await resp.json();
  const reply =
    json.choices?.[0]?.message?.content ??
    json.choices?.[0]?.content ??
    json.output_text ??
    JSON.stringify(json, null, 2);
  console.log("Model reply:\n", reply);
}

Model reply:
 The capital of France is **Paris**.


In [ ]:
// lc-github-models.ts
import { RunnableLambda } from "npm:@langchain/core/runnables";
import { AIMessage } from "npm:@langchain/core/messages";

function lcToGhMessages(input: any): Array<{ role: string; content: string }> {
  // 1) If the caller passed { messages }, use that; else { question }
  let msgs = input?.messages;
  if (!msgs && input?.question) {
    msgs = [{ role: "user", content: String(input.question) }];
  }

  // 2) If a prompt piped LC BaseMessages directly, input itself may be an array
  if (!msgs && Array.isArray(input)) {
    msgs = input;
  }

  if (!Array.isArray(msgs) || msgs.length === 0) {
    throw new Error("Provide { messages } or { question }.");
  }

  const mapRole = (t: string | undefined) => {
    const r = (t ?? "").toLowerCase();
    if (r === "human" || r === "user") return "user";
    if (r === "ai" || r === "assistant") return "assistant";
    if (r === "system") return "system";
    if (r === "developer") return "developer";
    // Fallback: if OpenAI-style already provided
    if (r === "tool" || r === "function") return "assistant"; // conservative fallback
    return "user";
  };

  const toText = (content: any): string => {
    if (typeof content === "string") return content;
    if (Array.isArray(content)) {
      // LC content parts: [{type:"text", text:"..."}, ...]
      return content
        .map((p) =>
          typeof p === "string"
            ? p
            : p?.text ?? p?.content ?? JSON.stringify(p)
        )
        .join("");
    }
    // Some LC BaseMessage store in .content/.lc_kwargs/etc.
    return content?.text ?? content?.content ?? String(content ?? "");
  };

  return msgs.map((m: any) => {
    const typeHint =
      m.role ?? m._type ?? m.type ?? m._getType?.() ?? m.constructor?.name;
    return {
      role: mapRole(typeHint),
      content: toText(m.content),
    };
  });
}

export function GitHubModelsChat(options?: {
  base?: string;
  org?: string;
  model?: string;
  token?: string | null;
  apiVersion?: string;
}) {
  const base = options?.base ?? Deno.env.get("GITHUB_MODELS_URL") ?? "https://models.github.ai";
  const org  = options?.org  ?? Deno.env.get("GITHUB_ORG") ?? undefined;
  const model= options?.model?? Deno.env.get("GITHUB_MODEL") ?? "openai/gpt-4.1";
  const token= options?.token?? Deno.env.get("GITHUB_TOKEN");
  const apiVersion = options?.apiVersion ?? "2022-11-28";

  if (!token) throw new Error("GITHUB_TOKEN is not set.");

  const endpoint = org
    ? `${base}/orgs/${org}/inference/chat/completions`
    : `${base}/inference/chat/completions`;

  return new RunnableLambda({
    name: "GitHubModelsChat",
    func: async (input: any) => {
      const messages = lcToGhMessages(input);

      const resp = await fetch(endpoint, {
        method: "POST",
        headers: {
          "Authorization": `Bearer ${token}`,
          "Accept": "application/json",
          "Content-Type": "application/json",
          "X-GitHub-Api-Version": apiVersion
        },
        body: JSON.stringify({ model, messages }),
      });

      if (!resp.ok) {
        const text = await resp.text();
        throw new Error(`GitHub Models request failed (${resp.status}): ${text}`);
      }

      const json = await resp.json();
      const reply =
        json.choices?.[0]?.message?.content ??
        json.choices?.[0]?.content ??
        json.output_text ??
        JSON.stringify(json, null, 2);

      return new AIMessage(reply);
    }
  });
}


In [25]:
import { ChatPromptTemplate } from "npm:@langchain/core/prompts";
import { StringOutputParser } from "npm:@langchain/core/output_parsers";
// import { GitHubModelsChat } from "./lc-github-models.ts";

const prompt = ChatPromptTemplate.fromMessages([
  ["system", "You are concise and accurate."],
  ["human", "{question}"],
]);

const model = GitHubModelsChat(); // uses env: GITHUB_TOKEN, GITHUB_MODEL, etc.
const toText = new StringOutputParser();

const chain = prompt.pipe(model).pipe(toText);

const answer = await chain.invoke({ question: "What is the capital of France?" });
console.log(answer); // "Paris"


The capital of France is Paris.


In [24]:
// hello_github_models_deno.ts
import { generateText } from "npm:ai";
import { createOpenAICompatible } from "npm:@ai-sdk/openai-compatible";

const token = Deno.env.get("GITHUB_TOKEN");
if (!token) throw new Error("GITHUB_TOKEN is not set.");

const github = createOpenAICompatible({
  name: "github",
  baseURL: "https://models.github.ai/inference", // GitHub Models endpoint
  headers: {
    Authorization: `Bearer ${token}`,
    "X-GitHub-Api-Version": "2022-11-28",
  },
});

const { text } = await generateText({
  model: github("openai/gpt-4o"), // or "openai/gpt-4.1", etc.
  prompt: "Say hello from GitHub Models in Deno!",
});

console.log(text);


Hello from GitHub Models in Deno! 🎉 🚀  
If you're running Deno with GitHub integrations, you're diving into some cutting-edge development — keep on building amazing things! 🌟


In [1]:
// deno run --allow-env --allow-net github_models_azure_sdk.ts

import ModelClient from "npm:@azure-rest/ai-inference";
import { AzureKeyCredential } from "npm:@azure/core-auth";

const token = Deno.env.get("GITHUB_TOKEN");
if (!token) throw new Error("GITHUB_TOKEN is not set.");

// 🔹 IMPORTANT: Use GitHub Models' endpoint, not Azure's
const endpoint = "https://models.github.ai/inference";

const client = new ModelClient(endpoint, new AzureKeyCredential(token));

// Pick any model from the GitHub Models Catalog, e.g.:
const model = "openai/gpt-4o"; // or "meta-llama/llama-3-8b-instruct"

const response = await client
  .path("/chat/completions") // Azure SDK still uses this path
  .post({
    body: {
      model,
      messages: [
        { role: "system", content: "You are a helpful assistant." },
        { role: "user", content: "Write a haiku about programming." },
      ],
    },
    headers: {
      "X-GitHub-Api-Version": "2022-11-28",
      Accept: "application/json",
    },
  });

if (response.status !== "200") {
  console.error("Error:", response.status, response.body);
} else {
  console.log("✅ GitHub Model Response:\n", response.body.choices[0].message.content);
}


✅ GitHub Model Response:
 Code flows like a stream,  
Logic dances line by line,  
Dreams in syntax bloom.


In [1]:
// chat.ts
import "npm:dotenv/config"; // Loads .env automatically (if using Node + dotenv)
import OpenAI from "npm:openai";

// Environment variables
const API_HOST = process.env.API_HOST ?? "github";

let client: OpenAI;
let MODEL_NAME: string;

switch (API_HOST) {
  case "azure": {
    // Example Azure setup - requires @azure/identity installed
    // and AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_CHAT_DEPLOYMENT in env.
    const { DefaultAzureCredential, getBearerTokenProvider } = await import(
      "@azure/identity"
    );

    const tokenProvider = getBearerTokenProvider(
      new DefaultAzureCredential(),
      "https://cognitiveservices.azure.com/.default"
    );

    client = new OpenAI({
      baseURL: process.env.AZURE_OPENAI_ENDPOINT,
      apiKey: tokenProvider as any, // Azure provides dynamic tokens
    });

    MODEL_NAME = process.env.AZURE_OPENAI_CHAT_DEPLOYMENT!;
    break;
  }

  case "ollama": {
    // For local Ollama models
    client = new OpenAI({
      baseURL: process.env.OLLAMA_ENDPOINT,
      apiKey: "nokeyneeded",
    });
    MODEL_NAME = process.env.OLLAMA_MODEL!;
    break;
  }

  case "github": {
    // GitHub Models endpoint — OpenAI-compatible
    client = new OpenAI({
      baseURL: "https://models.github.ai/inference",
      apiKey: process.env.GITHUB_TOKEN,
    });
    MODEL_NAME = process.env.GITHUB_MODEL ?? "openai/gpt-4o";
    break;
  }

  default: {
    // OpenAI.com
    client = new OpenAI({
      apiKey: process.env.OPENAI_KEY,
    });
    MODEL_NAME = process.env.OPENAI_MODEL!;
    break;
  }
}

// Make a chat completion request
const response = await client.chat.completions.create({
  model: MODEL_NAME,
  temperature: 0.7,
  messages: [
    {
      role: "system",
      content:
        "You are a helpful assistant that makes lots of cat references and uses emojis.",
    },
    { role: "user", content: "What's the weather in SF today?" },
  ],
});

console.log(`Response from ${API_HOST}:\n`);
console.log(response.choices[0].message.content);


Response from github:

Oh, I'm just a lil' data kitty and can't fetch real-time weather updates 😿. However, I can tell you that San Francisco weather usually likes to play peek-a-boo with the sun 🌥️🌞 — often cool, breezy, and a bit foggy (hello, Karl the Fog 🐾). Don’t forget to layer up, meow! 🧥😺

For today’s exact weather, check your favorite weather app or website. Purrhaps Weather.com or AccuWeather? 🐾


In [ ]:
import "npm:dotenv/config";
import { z } from "npm:zod";
import { ChatOpenAI } from "npm:@langchain/openai";
import { tool } from "npm:@langchain/core/tools";
import { AIMessage, HumanMessage, SystemMessage, ToolMessage } from "npm:@langchain/core/messages";

/** 1) PokéAPI tool (function-first signature) */
const getPokemon = tool(
  async ({ name }: { name: string }) => {
    const res = await fetch(`https://pokeapi.co/api/v2/pokemon/${encodeURIComponent(name)}`);
    if (!res.ok) throw new Error(`PokéAPI error: ${res.status}`);
    const d = await res.json();
    return {
      id: d.id,
      name: d.name,
      height: d.height,
      weight: d.weight,
      abilities: d.abilities.map((a: any) => a.ability.name),
      types: d.types.map((t: any) => t.type.name),
    };
  },
  {
    name: "get_pokemon",
    description: "Fetch basic info about a Pokémon (id, name, abilities, types).",
    schema: z.object({
      name: z.string().describe("Pokémon name in lowercase, e.g. 'pikachu'"),
    }),
  }
);

/** 2) GitHub Models via OpenAI-compatible API */
const llm = new ChatOpenAI({
  // must be your GitHub token
  apiKey: Deno.env.get("GITHUB_TOKEN") ?? process.env.GITHUB_TOKEN,

  // use `model`, not `modelName`
  model: Deno.env.get("GITHUB_MODEL") ?? process.env.GITHUB_MODEL ?? "openai/gpt-4o",

  // crucial: force the client to GitHub Models + required headers
  configuration: {
    baseURL: "https://models.github.ai/inference",
    defaultHeaders: {
      "X-GitHub-Api-Version": "2022-11-28",
      "Accept": "application/json",
    },
  },

  temperature: 0.2,
});



/** 3) Bind tool + 2-step tool-calling loop */
const modelWithTools = llm.bindTools([getPokemon]);

async function run() {
  const messages = [
    new SystemMessage("You are a helpful assistant. Use tools if needed."),
    new HumanMessage("Tell me about Pikachu."),
  ];

  const first = await modelWithTools.invoke(messages);

  if (!first.tool_calls?.length) {
    console.log(first.content);
    return;
  }

  const toolResponses: ToolMessage[] = [];
  for (const call of first.tool_calls) {
    if (call.name === "get_pokemon") {
      const result = await getPokemon.invoke(call.args);
      toolResponses.push(
        new ToolMessage({
          tool_call_id: call.id,
          name: call.name,
          content: JSON.stringify(result),
        })
      );
    }
  }

  const final = await modelWithTools.invoke([...messages, first as AIMessage, ...toolResponses]);
  console.log(final.content);
}

run().catch(console.error);


Promise { <pending> }

Pikachu is a Pokémon with the following characteristics:

- **ID**: 25
- **Type**: Electric
- **Height**: 0.4 meters (4 decimeters)
- **Weight**: 6.0 kilograms
- **Abilities**:
  - **Static**: May cause paralysis if the opponent makes physical contact.
  - **Lightning Rod**: Draws in all Electric-type moves to boost its Special Attack.

Pikachu is one of the most iconic Pokémon, known for its yellow fur and adorable appearance!


VoltAgent running at http://localhost:3141 (agent id: "assistant")




══════════════════════════════════════════════════
  VOLTAGENT SERVER STARTED SUCCESSFULLY
══════════════════════════════════════════════════
  ✓ HTTP Server:  http://localhost:3141
  ✓ Swagger UI:   http://localhost:3141/ui

  Test your agents with VoltOps Console: https://console.voltagent.dev
══════════════════════════════════════════════════
